In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

In [10]:
# --- 1. MEMUAT DATA ---

data = {
    'Temperature': [14.0, 39.0, 30.0, 38.0, 27.0, 32.0, -2.0, 3.0, 3.0, 28.0],
    'Humidity': [73, 96, 64, 83, 74, 55, 97, 85, 83, 74],
    'Wind Speed': [9.5, 8.5, 7.0, 1.5, 17.0, 3.5, 8.0, 6.0, 6.0, 8.5],
    'Precipitation (%)': [82.0, 71.0, 16.0, 82.0, 66.0, 26.0, 86.0, 96.0, 66.0, 107.0],
    'Cloud Cover': ['partly cloudy', 'partly cloudy', 'clear', 'clear', 'overcast', 'overcast', 'overcast', 'partly cloudy', 'overcast', 'clear'],
    'Atmospheric Pressure': [1010.82, 1011.43, 1018.72, 1026.25, 990.67, 1010.03, 990.87, 984.46, 999.44, 1012.13],
    'UV Index': [2, 7, 5, 7, 1, 2, 1, 1, 0, 8],
    'Season': ['Winter', 'Spring', 'Spring', 'Spring', 'Winter', 'Summer', 'Winter', 'Winter', 'Winter', 'Winter'],
    'Visibility (km)': [3.5, 10.0, 5.5, 1.0, 2.5, 5.0, 4.0, 3.5, 1.0, 7.5],
    'Location': ['inland', 'inland', 'mountain', 'coastal', 'mountain', 'inland', 'inland', 'inland', 'mountain', 'coastal'],
    'Weather Type': ['Rainy', 'Cloudy', 'Sunny', 'Sunny', 'Rainy', 'Cloudy', 'Snowy', 'Snowy', 'Snowy', 'Sunny']
}

df = pd.DataFrame(data)

In [15]:
# --- 2. PREPROCESSING DATA ---

# A. Pisahkan Fitur Kategorikal dan Numerik
numerical_features = ['Temperature', 'Humidity', 'Wind Speed', 'Precipitation (%)', 'Atmospheric Pressure', 'UV Index', 'Visibility (km)']
categorical_features = ['Cloud Cover', 'Season', 'Location']
target_label = 'Weather Type'

# B. Handle Fitur Kategorikal dan Mengubah 'Cloud Cover', 'Season', dan 'Location' menjadi kolom biner.
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# C. Pisahkan X (Fitur) dan Y (Label)
X = df_encoded.drop(columns=[target_label])
y = df_encoded[target_label]

# D. Normalisasi Fitur Numerik
scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])

print("✅ Data telah diproses dan dinormalisasi.")
print("\nBentuk data Features (X) setelah encoding dan normalisasi:", X.shape)


✅ Data telah diproses dan dinormalisasi.

Bentuk data Features (X) setelah encoding dan normalisasi: (10, 13)


In [16]:
# --- 3. MELATIH DAN MENGUJI MODEL k-NN ---

# A. Split Data (Training: 80%, Testing: 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)


# B. Mencari Nilai K Terbaik (Menggunakan Cross-Validation)
k_range = range(1, 6, 2) # Coba K=1, 3, 5 (ganjil), since n_samples_fit is 5
cv_scores = []
best_k = 1
highest_accuracy = 0.0

print("## 3. Mencari K Terbaik menggunakan Cross-Validation")
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Menggunakan cv=2 karena data hanya 10 baris
    scores = cross_val_score(knn, X, y, cv=2, scoring='accuracy')
    avg_score = scores.mean()
    cv_scores.append(avg_score)
    print(f"K={k}, Rata-rata Akurasi: {avg_score:.4f}")

    if avg_score > highest_accuracy:
        highest_accuracy = avg_score
        best_k = k

print(f"\n🏆 Nilai K terbaik adalah K={best_k} (Akurasi Rata-rata: {highest_accuracy:.4f})")
print("-" * 60)


# C. Evaluasi Model Akhir
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train, y_train)

y_pred = knn_final.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print("## 4. Evaluasi Model Akhir pada Data Test (20%)")
print(f"Prediksi Data Test:\n{y_pred}")
print(f"\nHasil Sebenarnya Data Test:\n{y_test.values}")
print(f"\nFinal Accuracy (K={best_k}): {final_accuracy:.2f}")

## 3. Mencari K Terbaik menggunakan Cross-Validation
K=1, Rata-rata Akurasi: 0.5000
K=3, Rata-rata Akurasi: 0.3000
K=5, Rata-rata Akurasi: 0.2000

🏆 Nilai K terbaik adalah K=1 (Akurasi Rata-rata: 0.5000)
------------------------------------------------------------
## 4. Evaluasi Model Akhir pada Data Test (20%)
Prediksi Data Test:
['Snowy' 'Snowy' 'Sunny' 'Sunny']

Hasil Sebenarnya Data Test:
['Snowy' 'Rainy' 'Cloudy' 'Sunny']

Final Accuracy (K=1): 0.50
